#البنية التقنية المتكاملة لنظام CardioCoin
نظرة عامة على المشروع
هذا النظام مصمم لمراقبة صحة القلب واكتشاف السقوط بشكل ذكي، حيث يدمج بين تقنيات التعلم العميق والبيانات الطبية الحيوية لتوفير حماية فورية للمستخدمين.

المكونات الرئيسية للبنية البرمجية:
1. المعالجة الديناميكية للبيانات (Dynamic Data Processing):

تم برمجة منطق خاص لتحديد "عتبات النبض" بناءً على عمر المستخدم (أطفال، بالغين، كبار سن) لضمان دقة التشخيص.

2. النمذجة الهجينة للذكاء الاصطناعي (Hybrid AI Modeling):

نموذج CNN-LSTM: يستخدم لتحليل الإشارات الزمنية للنبض وحركة الجسم معاً لتصنيف الحالة الحالية للمستخدم.

نموذج Autoencoder: يعمل كأداة لاكتشاف الشذوذ (Anomaly Detection) لضمان جودة البيانات المرسلة من المستشعرات واكتشاف أي خلل مفاجئ.

3. الربط السحابي والتحليل اللغوي (Cloud & LLM Integration):

في حالة رصد خطر (سقوط أو اضطراب نبض)، يقوم النظام بالتواصل مع منصة "عِلم" لاستخدام نماذج اللغة الكبيرة في صياغة رسائل طوارئ إنسانية ودقيقة.

#1. استيراد المكتبات الأساسية
في هذه الخلية، نقوم باستدعاء المكتبات اللازمة لمعالجة البيانات الرياضية (Numpy)، بناء نماذج الذكاء الاصطناعي (TensorFlow)، والربط مع الخدمات السحابية (Requests).

In [56]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras import Model
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output

In [57]:
from openai import OpenAI

In [58]:
import requests
import time

#2. إعدادات الوصول والربط السحابي
هنا يتم وضع مفاتيح الوصول (API Keys) الخاصة بمنصة "عِلم" لتمكين النظام من إرسال التنبيهات الذكية عبر نماذج اللغة الكبيرة (LLM).

In [59]:
def ask_nuha_direct(prompt):
    # استخدام المفتاح
    api_key = "sk-3GioTnUIug01Ny3zN50IEg"
    url = "https://elmodels.ngrok.app/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "nuha-2.0",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }

    # محاولة الاتصال مرتين في حال حدوث Timeout
    for attempt in range(2):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)

            if response.status_code == 200:
                return response.json()['choices'][0]['message']['content']
            else:
                return f"⚠️ استجابة غير متوقعة ({response.status_code})"

        except requests.exceptions.Timeout:
            if attempt == 0:
                print("🔄 الخادم بطيء، جاري المحاولة مرة أخرى...")
                time.sleep(2)
                continue
            return "❌ الخادم لا يستجيب حالياً، يرجى المحاولة بعد قليل."
        except Exception as e:
            return f"⚠️ خطأ فني: {str(e)}"

In [60]:
print(ask_nuha_direct("المريض في حالة خطر، صغ رسالة طوارئ."))# تاكيد استجابة التقنية

أنا **نهى**، مساعدك الذكي من شركة **علم**.

بما أن المريض في حالة خطر، يجب عليك **الاتصال فوراً برقم الطوارئ (997 في مصر، أو 911 في الولايات المتحدة، أو 999 في المملكة المتحدة، أو رقم الطوارئ المحلي في دولتك)**.

إليك نموذج لرسالة طوارئ يمكنك إرسالها فوراً إلى شخص قريب منك أو إلى خدمة الطوارئ عبر الرسائل النصية (SMS) إذا كان الاتصال الصوتي غير متاح:

***

**نموذج الرسالة:**

"⚠️ **تنبيه طارئ جداً** ⚠️
المريض في حالة حرجة وخطرة.
📍 **الموقع:** [أدخل العنوان الدقيق هنا: اسم الشارع، رقم المنزل، أو نقطة مرجعية واضحة].
👤 **حالة المريض:** [أدخل الأعراض باختصار: توقف تنفس، نزيف، ألم صدر، فقدان وعي...].
🆘 **تحتاج إسعافاً فورياً.**
📞 **رقم الاتصال:** [أدخل رقم هاتفك].
سأبقى في مكان انتظاركم."

***

**تنبيه هام:**
*   تأكد من بقاء خط الاتصال مفتوحاً أو انتظر وصول المساعدة في مكان آمن.
*   إذا كان المريض لا يتنفس، ابدأ **الإنعاش القلبي الرئوي (CPR)** فوراً إذا كنت مدرباً على ذلك.

أتمنى الشفاء العاجل للمريض، وتوكل على الله.


In [61]:
def ask_nuha_integrated(age, status, pulse):
    # هنا نصيغ التعليمات بدقة لنهى
    system_instruction = (
        "أنتِ المساعد الذكي لنظام CardioCoin الطبي. "
        "مهمتكِ صياغة رسالة طوارئ قصيرة جداً (SMS) ليتم إرسالها للمسعفين. "
        "يجب أن تتضمن الرسالة: نوع الخطر، عمر المريض، ونبض القلب الحالي. "
        "اجعليها احترافية ومباشرة بدون مقدمات طويلة."
    )

    user_prompt = f"المريض عمره {age}، رصد النظام حالة {status} بنبض {pulse} نبضة/دقيقة."

    full_prompt = f"{system_instruction}\n\n{user_prompt}"

    # استدعاء الداله
    return ask_nuha_direct(full_prompt)

# تجربة التشغيل لرؤية الفرق
print(ask_nuha_integrated(age=72, status="سقوط مفاجئ (Fall Risk)", pulse=115))

طوارئ طبية: سقوط مفاجئ. عمر المريض: 72 سنة. نبض القلب: 115/دقيقة. يرجى الحضور العاجل. - نظام CardioCoin


#3. تحديد المعايير الحيوية حسب الفئة العمرية
هذه الدالة هي المسؤول الذكي عن تحديد "النبض الطبيعي". النظام هنا يميز تلقائياً بين الطفل، البالغ، وكبير السن لضمان دقة التشخيص وعدم إعطاء إنذارات كاذبة

In [62]:
def set_heart_rate_thresholds(age):
    if age < 12:
        return {"low": 70, "high": 110}  #نبضات الأطفال
    elif age >= 65:
        return {"low": 50, "high": 90}   # نبضات كبار السن (إضافة اختيارية للتميز)
    else:
        return {"low": 60, "high": 100}  # نبضات البالغين

#4. بناء المعمارية الذكية للنظام (AI Models)
نقوم هنا بتصميم نموذجين:

CNN-LSTM: لتحليل تتابع نبضات القلب وحركة الجسم وتصنيف الحالات.

Autoencoder: كطبقة أمان إضافية لاكتشاف أي شذوذ غير معتاد في الإشارات الحيوية.

In [63]:
# أ. نموذج CNN-LSTM (المشخص الرئيسي)
def build_cardio_model(input_shape=(100, 2)):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        Conv1D(64, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(3, activation='softmax') # التصنيفات: Normal, Exercise, Fall
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [64]:
# ب. نموذج Autoencoder (مدقق جودة الإشارة واكتشاف الشذوذ)
def build_anomaly_detector(input_dim=100):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(32, activation='relu')(input_layer)
    decoded = Dense(100, activation='sigmoid')(encoded)
    auto_model = Model(input_layer, decoded)
    auto_model.compile(optimizer='adam', loss='mse')
    return auto_model

#5. محاكاة وتجهيز بيانات التدريب
نقوم بتوليد بيانات رقمية تحاكي إشارات القلب وحركة الجسم في حالات (الاستقرار، الرياضة، والسقوط) لتدريب النماذج عليها.

In [65]:

# محاكاة بيانات التدريب
X_train = np.random.randn(100, 100, 2)
y_train = tf.keras.utils.to_categorical(np.random.randint(0, 3, 100), 3)

# التدريب
print("🔄 جاري تدريب النماذج الذكية...")
cardio_model = build_cardio_model()
cardio_model.fit(X_train, y_train, epochs=2, verbose=0)
print("✅ تم تدريب جميع النماذج بنجاح.")

🔄 جاري تدريب النماذج الذكية...
✅ تم تدريب جميع النماذج بنجاح.


In [66]:
# توليد بيانات افتراضية تحاكي الواقع لتدريب النماذج
num_samples = 1000
seq_length = 100

# إنشاء بيانات نبض وحركة عشوائية ومنظمة
X_train_pulse = np.random.randn(num_samples, seq_length)
X_train_motion = np.random.randn(num_samples, seq_length)
X_train = np.stack([X_train_pulse, X_train_motion], axis=-1)

# إنشاء تصنيفات عشوائية (0: Normal, 1: Exercise, 2: Fall)
y_train = tf.keras.utils.to_categorical(np.random.randint(0, 3, num_samples), num_classes=3)

print("✅ تم تجهيز بيانات التدريب.")

✅ تم تجهيز بيانات التدريب.


#  المحرك التشغيلي والربط السحابي.6

In [67]:
def run_cardio_coin_system(pulse_rate, motion_data, age):
    # أولاً: تحديد بناءً على العمر
    limits = set_heart_rate_thresholds(age)

    # ثانياً: فحص الحالة
    # محاكاة لقرار الموديل الذكي بناءً على البيانات
    # Ensure pulse_rate is an array of the same length as motion_data for np.stack
    pulse_rate_array = np.full(len(motion_data), pulse_rate)
    input_signal = np.stack([pulse_rate_array, motion_data], axis=-1).reshape(1, 100, 2)

    # قرار افتراضي (يُستبدل بـ model.predict في البيئة الفعلية)
    if pulse_rate > limits["high"] and np.mean(motion_data) < 1.0:
        status = "Fall Risk"
    elif pulse_rate < limits["low"]:
        status = "Bradycardia"
    else:
        status = "Stable"

    # ثالثاً: التواصل مع منصة عِلم في حالات الخطر
    if status in ["Fall Risk", "Bradycardia"]:
        prompt = f"المريض عمره {age} وحالته {status}. صغ رسالة طوارئ ذكية فورية."
        try:
            headers = {"Authorization": f"Bearer {ELM_API_KEY}", "Content-Type": "application/json"}
            payload = {"model": "llm", "messages": [{"role": "user", "content": prompt}]}
            response = requests.post(f"{BASE_URL}/chat/completions", headers=headers, json=payload, timeout=5)
            return response.json()['choices'][0]['message']['content']
        except:
            return f"⚠️ تنبيه محلي عاجل: تم رصد حالة {status}."

    return f"💚 نبض المريض ضمن النطاق الطبيعي لعمره ({age} سنة)."

In [68]:
# مثال لمريض بالغ بنبض 110 (يعتبر مرتفع)
print("اختبار البالغ:")
print(run_cardio_coin_system(110, [0.1]*100, age=30))

# مثال لطفل بنفس النبض 110 (يعتبر طبيعي)
print("\nاختبار الطفل:")
print(run_cardio_coin_system(110, [0.1]*100, age=10))

اختبار البالغ:
⚠️ تنبيه محلي عاجل: تم رصد حالة Fall Risk.

اختبار الطفل:
💚 نبض المريض ضمن النطاق الطبيعي لعمره (10 سنة).


#7. مرحلة التدريب الفعلي (Training)
في هذه الخطوة، يتم تعليم النماذج الذكية كيفية التعرف على الأنماط المختلفة للبيانات الحيوية لضمان استجابة دقيقة عند الاستخدام الفعلي.

In [69]:
# بناء الموديل الرئيسي
cardio_model = build_cardio_model()

# بدء التدريب الفعلي
print("🔄 جاري تدريب الموديل الرئيسي (CNN-LSTM)...")
cardio_model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

# بناء وتدريب كاشف الشذوذ (Autoencoder)
anomaly_model = build_anomaly_detector(input_dim=seq_length)
anomaly_model.fit(X_train_pulse, X_train_pulse, epochs=5, verbose=0)

print("✅ تم تدريب جميع النماذج بنجاح.")

🔄 جاري تدريب الموديل الرئيسي (CNN-LSTM)...
✅ تم تدريب جميع النماذج بنجاح.


#8. نظام التحكم المتكامل وبروتوكول الطوارئ
هذه هي الخلية الرئيسية التي تربط كل شيء ببعضه؛ حيث تستقبل البيانات، تحللها بناءً على العمر والحركة، وفي حال رصد خطر (سقوط أو اضطراب نبض)، تتواصل مع السحابة لصياغة رسالة طوارئ فورية.

In [74]:
def run_cardio_coin_system(pulse_rate, motion_data, age):
    # 1. تحديد ديناميكية بناءً على العمر
    limits = set_heart_rate_thresholds(age)

    # 2. فحص النبض والحركة (Logic Layer)
    if pulse_rate > limits["high"]:
        # استخدام متوسط الحركة (Mean) للتفريق بين الرياضة والسقوط
        if np.mean(motion_data) < 1.0:
            status = "Fall Risk / High Distress"
        else:
            status = "Stable - Exercise"
    elif pulse_rate < limits["low"]:
        status = "Bradycardia (Low Pulse)"
    else:
        status = "Stable"

    # 3. الربط الذكي مع نهى 2.0 في حالات الطوارئ
    if status in ["Fall Risk / High Distress", "Bradycardia (Low Pulse)"]:
        # استدعاء الدالة المتكاملة التي ضبطناها سوياً
        try:
            return ask_nuha_integrated(age, status, pulse_rate)
        except:
            # رسالة طوارئ احتياطية في حال فشل الإنترنت تماماً
            return f"🚨 تنبيه محلي عاجل: تم رصد حالة {status} للمريض."

    return f"💚 الحالة مستقرة: {status} (العمر: {age})"

In [71]:
def test_cardiocoin_system():
    # سنختبر 3 حالات واقعية
    test_scenarios = [
        {"name": "حالة سقوط مسن", "age": 75, "pulse": 110, "motion": 0.2}, # نبض عالٍ وحركة شبه منعدمة
        {"name": "نشاط رياضي لشاب", "age": 22, "pulse": 135, "motion": 5.5}, # نبض عالٍ وحركة قوية
        {"name": "حالة انخفاض نبض (طفل)", "age": 8, "pulse": 55, "motion": 1.2}  # نبض منخفض جداً لطفل
    ]

    print("🚀 بدء اختبار نظام CardioCoin المتكامل...\n")
    print("-" * 50)

    for case in test_scenarios:
        print(f"📋 التجربة: {case['name']}")

        # 1. تشغيل المحرك التقني لتصنيف الحالة
        status = "Fall Risk / High Distress" if case['pulse'] > 100 and case['motion'] < 1.0 else "Stable - Exercise"
        if case['pulse'] < 60: status = "Bradycardia (Low Pulse)"

        # 2. استدعاء نهى في حالات الخطر فقط
        if "Stable" not in status:
            print(f"🔍 النظام رصد: {status}")
            print("🤖 جاري صياغة رسالة الطوارئ عبر 'نهى 2.0'...")

            # استدعاء الدالة
            emergency_msg = ask_nuha_integrated(case['age'], status, case['pulse'])
            print(f"📩 الرسالة الناتجة: {emergency_msg}")
        else:
            print(f"✅ الحالة: {status} - لا يتطلب تدخل الطوارئ.")

        print("-" * 50)

# تشغيل الاختبار الشامل
test_cardiocoin_system()

🚀 بدء اختبار نظام CardioCoin المتكامل...

--------------------------------------------------
📋 التجربة: حالة سقوط مسن
🔍 النظام رصد: Fall Risk / High Distress
🤖 جاري صياغة رسالة الطوارئ عبر 'نهى 2.0'...
📩 الرسالة الناتجة: توطين طارئ: سقوط محتمل (Fall Risk) مع ضيق تنفسي حاد.
المريض: 75 سنة.
نبض القلب: 110/دقيقة.
تحتاج عناية فورية.
نظام CardioCoin - شركة علم.
--------------------------------------------------
📋 التجربة: نشاط رياضي لشاب
✅ الحالة: Stable - Exercise - لا يتطلب تدخل الطوارئ.
--------------------------------------------------
📋 التجربة: حالة انخفاض نبض (طفل)
🔍 النظام رصد: Bradycardia (Low Pulse)
🤖 جاري صياغة رسالة الطوارئ عبر 'نهى 2.0'...
📩 الرسالة الناتجة: تحذير طوارئ: حالة Bradycardia (انخفاض نبض). عمر المريض: 8 سنوات. النبض الحالي: 55/دقيقة. يرجى التوجه فوراً.
--------------------------------------------------


In [72]:
def run_cardio_coin_system(pulse, motion_mean, age):
    limits = set_heart_rate_thresholds(age)
    status = "Stable"

    if pulse > limits["high"]:
        status = "Fall Risk / High Distress" if motion_mean < 1.0 else "Stable - Exercise"
    elif pulse < limits["low"]:
        status = "Bradycardia (Low Pulse)"

    if "Stable" not in status:
        print(f"🔍 النظام رصد: {status}")
        return ask_nuha_integrated(age, status, pulse)
    return f"💚 الحالة مستقرة (العمر: {age})"

# تجربة النظام
print("🚀 بدء اختبار CardioCoin:")
print(f"حالة المسن: {run_cardio_coin_system(110, 0.2, 75)}")

🚀 بدء اختبار CardioCoin:
🔍 النظام رصد: Fall Risk / High Distress
حالة المسن: تنبيه طوارئ: حالة سقوط محتملة مع ضيق تنفسي شديد. المريض: 75 عاماً. النبض: 110 ض/د. تطلب التدخل الفوري. - نظام CardioCoin


In [73]:
# إعداد أدوات الواجهة
age_slider = widgets.IntSlider(value=25, min=1, max=100, description='العمر:')
pulse_slider = widgets.IntSlider(value=75, min=30, max=200, description='النبض:')
motion_dropdown = widgets.Dropdown(
    options=[('سقوط أو إغماء', 0.2), ('مش طبيعي', 1.5), ('جري/رياضة', 5.0)],
    value=1.5,
    description='الحركة:',
)
button = widgets.Button(description="⚡ فحص الحالة الآن", button_style='danger')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        print("🔍 جاري تحليل البيانات...")
        # استدعاء المحرك التشغيلي
        result = run_cardio_coin_system(pulse_slider.value, motion_dropdown.value, age_slider.value)
        print(f"\n{result}")

button.on_click(on_button_clicked)

# عرض الواجهة
print("🛠️ لوحة اختبار نظام CardioCoin:")
display(age_slider, pulse_slider, motion_dropdown, button, output)

🛠️ لوحة اختبار نظام CardioCoin:


IntSlider(value=25, description='العمر:', min=1)

IntSlider(value=75, description='النبض:', max=200, min=30)

Dropdown(description='الحركة:', index=1, options=(('سقوط أو إغماء', 0.2), ('مش طبيعي', 1.5), ('جري/رياضة', 5.0…

Button(button_style='danger', description='⚡ فحص الحالة الآن', style=ButtonStyle())

Output()